# TED 03 — StudyTrack IA | Classificação multiclasse de `grade`

**Caderno de execução (Google Colab)** · Big Data e Ciências de Dados · ADS 2026.2

**Objetivo:** comparar Regressão Logística, Árvore de Decisão e Random Forest usando a base preparada na TED 02. O texto acadêmico cumulativo pertence ao **Documento Técnico v3.0**, em arquivo separado.

> As métricas da validação são exploratórias até conferência do grupo. O teste final permanece bloqueado até a seleção fundamentada.


## 1. Configurações aprovadas

| Item | Configuração |
|---|---|
| Alvo | `grade`, oito classes (0 a 7) |
| Preditoras | 31 variáveis |
| Treino/teste | 80/20 estratificado |
| Validação | Cinco partes, semente 42 |
| Critério principal | F1-macro; acurácia complementar |
| Codificação | 16 nominais por One-Hot; 15 ordinais ordenadas |
| Seleção | Somente pela validação cruzada no treinamento |


## 2. Bibliotecas e leitura da base

Fonte: [UCI — Higher Education Students Performance Evaluation](https://doi.org/10.24432/C51G82). CSV preparado na TED 02, sem identificador pessoal.


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)

SEMENTE = 42
plt.rcParams.update({"figure.figsize": (8, 4.8), "axes.grid": False, "font.size": 10})

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)


In [ ]:
URL_DADOS = (
    "https://raw.githubusercontent.com/Francisco-alt-coder/"
    "StudyTrack_IA/61d910e73b19f868b9e234cbd92c8756216a53a9/"
    "data/processed/dataset_modelagem.csv"
)

dados = pd.read_csv(URL_DADOS)
print("Dimensão da base:", dados.shape)
display(dados.head(3))


In [ ]:
alvo = "grade"
assert alvo in dados.columns, "A coluna-alvo não foi encontrada."

X = dados.drop(columns=[alvo])
y = dados[alvo]

assert X.shape[1] == 31, "As 31 variáveis aprovadas não foram encontradas."
assert set(y.unique()) == set(range(8)), "As classes de grade diferem das documentadas."

qualidade = pd.DataFrame({
    "verificacao": ["Registros", "Preditoras", "Classes", "Valores ausentes", "Duplicados"],
    "resultado": [
        len(dados), X.shape[1], y.nunique(),
        int(dados.isna().sum().sum()), int(dados.duplicated().sum())
    ]
})
display(qualidade)
print("Tipos de dados presentes:", X.dtypes.value_counts().to_dict())


### 2.1 Distribuição das classes

Conferir a frequência das oito classes antes da modelagem.


In [ ]:
rotulos_grade = {0: "Fail", 1: "DD", 2: "DC", 3: "CC", 4: "CB", 5: "BB", 6: "BA", 7: "AA"}
frequencias = y.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4.6))
barras = ax.bar(
    [rotulos_grade[classe] for classe in frequencias.index],
    frequencias.values, color="#3869a6"
)
ax.bar_label(barras, padding=3)
ax.set_title("Distribuição das oito classes de desempenho")
ax.set_xlabel("Classe de grade")
ax.set_ylabel("Número de estudantes")
ax.set_ylim(0, int(frequencias.max()) + 7)
fig.tight_layout()
plt.show()

display(pd.DataFrame({
    "classe": [rotulos_grade[c] for c in frequencias.index],
    "registros": frequencias.values,
    "percentual": (frequencias.values / len(y) * 100).round(1)
}))


## 3. Divisão estratificada 80/20

Manter o conjunto de teste reservado. A validação cruzada utilizará apenas o treinamento.


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEMENTE,
    stratify=y
)

assert set(X_treino.index).isdisjoint(set(X_teste.index))
assert len(X_treino) + len(X_teste) == len(X)
print(f"Treinamento: {len(X_treino)} registros")
print(f"Teste reservado: {len(X_teste)} registros")


In [ ]:
distribuicao = pd.DataFrame({
    "base_total": y.value_counts().sort_index(),
    "treinamento": y_treino.value_counts().sort_index(),
    "teste_reservado": y_teste.value_counts().sort_index()
}).fillna(0).astype(int)

distribuicao.index.name = "classe_grade"
display(distribuicao)
print("Menor classe no treinamento:", int(y_treino.value_counts().min()))


## 4. Tratamento misto das variáveis

**Nominais:** One-Hot Encoding. **Ordinais:** códigos ordenados; padronização apenas na Regressão Logística. As transformações serão ajustadas dentro de cada pipeline.


In [ ]:
variaveis_nominais = [
    "sexo", "tipo_ensino_medio", "trabalho_adicional",
    "atividade_artistica_esportiva", "possui_parceiro",
    "transporte_universidade", "tipo_moradia", "situacao_pais",
    "ocupacao_mae", "ocupacao_pai", "seminarios_conferencias",
    "impacto_projetos_atividades", "preparacao_prova_companhia",
    "preparacao_prova_momento", "sala_invertida", "course_id"
]

variaveis_ordinais = [
    "idade", "tipo_bolsa", "faixa_renda", "escolaridade_mae",
    "escolaridade_pai", "numero_irmaos", "horas_estudo_semanais",
    "leitura_nao_cientifica", "leitura_cientifica",
    "frequencia_aulas", "anotacoes_aula", "atencao_aula",
    "discussao_interesse_sucesso", "gpa_semestre_anterior",
    "gpa_esperado_formatura"
]

assert len(variaveis_nominais) == 16
assert len(variaveis_ordinais) == 15
assert not (set(variaveis_nominais) & set(variaveis_ordinais))
assert set(variaveis_nominais + variaveis_ordinais) == set(X.columns)

print("Mapeamento aprovado: 16 nominais e 15 ordinais.")


## 5. Pipelines independentes

Regressão Logística, Árvore de Decisão e Random Forest. Parâmetros iniciais de referência, sem ajuste automático de hiperparâmetros.


In [ ]:
def criar_preprocessador_linear():
    return ColumnTransformer([
        ("nominais", OneHotEncoder(handle_unknown="ignore", sparse_output=False), variaveis_nominais),
        ("ordinais", StandardScaler(), variaveis_ordinais)
    ], remainder="drop")

def criar_preprocessador_arvores():
    return ColumnTransformer([
        ("nominais", OneHotEncoder(handle_unknown="ignore", sparse_output=False), variaveis_nominais),
        ("ordinais", "passthrough", variaveis_ordinais)
    ], remainder="drop")

modelos = {
    "Regressão Logística": Pipeline([
        ("preprocessamento", criar_preprocessador_linear()),
        ("classificador", LogisticRegression(max_iter=3000, random_state=SEMENTE))
    ]),
    "Árvore de Decisão": Pipeline([
        ("preprocessamento", criar_preprocessador_arvores()),
        ("classificador", DecisionTreeClassifier(random_state=SEMENTE))
    ]),
    "Random Forest": Pipeline([
        ("preprocessamento", criar_preprocessador_arvores()),
        ("classificador", RandomForestClassifier(random_state=SEMENTE, n_jobs=1))
    ])
}

print("Modelos configurados:", ", ".join(modelos))


## 6. Validação cruzada estratificada

Cinco partes no treinamento. Registrar média e desvio-padrão de F1-macro e acurácia, além do F1 no treinamento para observar possível sobreajuste.


In [ ]:
validacao = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEMENTE
)

pontuacoes = {}
for nome, pipeline in modelos.items():
    resultado = cross_validate(
        pipeline,
        X_treino, y_treino,
        cv=validacao,
        scoring={"f1_macro": "f1_macro", "acuracia": "accuracy"},
        return_train_score=True,
        n_jobs=1,
        error_score="raise"
    )
    pontuacoes[nome] = resultado
    print(f"Validação concluída: {nome}")


In [ ]:
linhas = []
for nome, resultado in pontuacoes.items():
    linhas.append({
        "modelo": nome,
        "f1_macro_medio": resultado["test_f1_macro"].mean(),
        "f1_macro_dp": resultado["test_f1_macro"].std(ddof=1),
        "acuracia_media": resultado["test_acuracia"].mean(),
        "acuracia_dp": resultado["test_acuracia"].std(ddof=1),
        "f1_treino_medio": resultado["train_f1_macro"].mean()
    })

resumo_cv = pd.DataFrame(linhas).sort_values(
    "f1_macro_medio", ascending=False
).reset_index(drop=True)

display(resumo_cv.round(3))


### 6.1 Comparação gráfica

Barras de erro representam o desvio-padrão entre as cinco partes, não um intervalo de confiança.


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.7), sharey=True)

indicadores = [
    ("f1_macro_medio", "f1_macro_dp", "F1-macro"),
    ("acuracia_media", "acuracia_dp", "Acurácia")
]
for eixo, (media, desvio, titulo) in zip(eixos, indicadores):
    eixo.barh(
        resumo_cv["modelo"], resumo_cv[media],
        xerr=resumo_cv[desvio],
        color="#3869a6", alpha=0.90, capsize=4
    )
    eixo.set_xlim(0, 1)
    eixo.set_xlabel("Pontuação")
    eixo.set_title(f"{titulo}: média ± desvio-padrão")

eixos[0].invert_yaxis()

fig.suptitle("Comparação dos classificadores — validação cruzada (treinamento)")
fig.tight_layout()
plt.show()


## 7. Escolha do classificador pelo grupo

Após conferir a tabela e os gráficos: qual modelo teve maior F1-macro? Os resultados variaram entre as partes? Houve diferença grande entre treino e validação?

**Modelo aprovado:** [preencher após discussão]  
**Justificativa e responsáveis:** [preencher]  
**Data da aprovação:** [preencher]

A célula seguinte mostra o maior F1-macro como referência, mas **não escolhe automaticamente**.


In [ ]:
modelo_mais_bem_pontuado_cv = resumo_cv.loc[0, "modelo"]
print("Maior F1-macro médio na validação:", modelo_mais_bem_pontuado_cv)
print("Essa indicação não substitui a decisão fundamentada do grupo.")

# Depois da reunião e da análise dos resultados, escrever
# exatamente um dos nomes existentes no dicionário 'modelos'.
# Manter None até a decisão ser registrada.
MODELO_APROVADO = None

if MODELO_APROVADO is None:
    print("Avaliação final do teste bloqueada até a aprovação do grupo.")
else:
    assert MODELO_APROVADO in modelos, "Nome do modelo não reconhecido."


## 8. Teste final — somente após a aprovação

Alterar `MODELO_APROVADO = None` para o nome definido pelo grupo, após registrar a justificativa na seção anterior. Executar a avaliação final apenas uma vez.


In [ ]:
modelo_final = None
previsoes_teste = None

if MODELO_APROVADO is None:
    print("Teste ainda não executado: aguarde a aprovação do grupo.")
else:
    modelo_final = clone(modelos[MODELO_APROVADO])
    modelo_final.fit(X_treino, y_treino)
    previsoes_teste = modelo_final.predict(X_teste)

    print("Modelo avaliado:", MODELO_APROVADO)
    print("F1-macro (teste):", round(f1_score(y_teste, previsoes_teste, average="macro", zero_division=0), 3))
    print("Acurácia (teste):", round(accuracy_score(y_teste, previsoes_teste), 3))
    print("\nRelatório por classe:")
    print(classification_report(
        y_teste, previsoes_teste,
        labels=list(rotulos_grade),
        target_names=list(rotulos_grade.values()),
        zero_division=0
    ))


In [ ]:
if previsoes_teste is None:
    print("Matriz de confusão indisponível até a avaliação autorizada.")
else:
    fig, ax = plt.subplots(figsize=(8.5, 7))
    ConfusionMatrixDisplay.from_predictions(
        y_teste, previsoes_teste,
        labels=list(rotulos_grade),
        display_labels=list(rotulos_grade.values()),
        cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(f"Matriz de confusão no teste — {MODELO_APROVADO}")
    fig.tight_layout()
    plt.show()


## 9. Conferência dos resultados finais

**F1-macro e acurácia:** [conferir]  
**Classes com mais erros:** [conferir matriz de confusão]  
**Limitações e possíveis vieses:** [analisar no Documento Técnico v3.0].

O conjunto possui poucas observações nas classes minoritárias; evitar conclusões generalizadas.


## 10. Registro de execução e autoria

- **Quem executou e em qual data:** [preencher]
- **Versões das bibliotecas:** [conferir saída da primeira célula]
- **Quem conferiu as métricas e gráficos:** [preencher]
- **Como foi escolhida a opção final:** [registrar antes do teste]
- **Histórico de uso da IA e validação pelo grupo:** [anexar conforme orientação do professor]

A redação acumulativa das TEDs 01, 02 e 03 deve ficar no **Documento Técnico v3.0**, não neste notebook.
